# Verificar lo reconstruido: mapas de diferencia y modelos de análisis

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e Inteligencia Artificial — IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento Digital de Imágenes
* **Nombre de trabajo:** Laboratorio de Tecnologías de la Imagen Digital

---

## Objetivo

Usar modelos de visión para analizar fotos restauradas y reconocer qué pueden y qué no pueden comprobar.

## Al terminar este cuaderno vas a poder:

1. Dibujar un mapa de diferencias entre original y restauración.
2. Clasificar y detectar objetos con modelos de Hugging Face sin entrenarlos.
3. Explicar por qué un modelo de análisis no detecta una invención creíble.

## Preparación

Este cuaderno **usa los resultados** de los cuadernos 03 y 04 (guardados en `resultados/` en tu Drive). Si no los ejecutaste, hacelo primero.

Los dos modelos de análisis (`SigLIP` y `RT-DETR`) son livianos y corren en la T4.

In [ ]:
%pip install -q transformers accelerate

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

CARPETA = '/content/drive/MyDrive/015_modelos_difusion'  # carpeta de la unidad en tu Drive
DATOS = f'{CARPETA}/datos'  # imágenes de ejemplo

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Sin GPU. Menú: Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU')

print('GPU:', torch.cuda.get_device_name(0))
print('Memoria de la GPU:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GB')

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

RESULTADOS = f'{CARPETA}/resultados'

necesarios = [
    'perm_referencia.png', 'perm_bicubica.png', 'perm_difusion_x4.png',
    'retrato_original.png', 'retrato_ojos_semilla0.png', 'retrato_ojos_semilla1.png', 'retrato_ojos_semilla2.png',
]
faltan = [n for n in necesarios if not os.path.exists(f'{RESULTADOS}/{n}')]
if faltan:
    raise FileNotFoundError(f'Ejecutá primero los cuadernos 03 y 04. Faltan: {faltan}')

referencia = Image.open(f'{RESULTADOS}/perm_referencia.png').convert('RGB')
bicubica = Image.open(f'{RESULTADOS}/perm_bicubica.png').convert('RGB')
difusion = Image.open(f'{RESULTADOS}/perm_difusion_x4.png').convert('RGB')
print('Resultados cargados.')

## 1. Mapa de diferencias

Restamos cada versión al original y mostramos **dónde** cambió, en una escala de color: oscuro, casi igual; claro, muy distinto. Es la herramienta más simple y más honesta para mostrar qué tocó el modelo.

In [ ]:
ref = np.array(referencia).astype(float)
pares = [('bicúbica', bicubica), ('difusión x4', difusion)]

figura, ejes = plt.subplots(1, 2, figsize=(12, 5.5))
for numero in range(len(pares)):
    nombre, imagen = pares[numero]
    diferencia = np.abs(ref - np.array(imagen).astype(float)).mean(axis=2)  # promedio de los 3 canales
    mapa = ejes[numero].imshow(diferencia, cmap='magma', vmin=0, vmax=60)
    ejes[numero].set_title(f'{nombre}: dónde difiere del original')
    ejes[numero].axis('off')
    cambiaron = (diferencia > 10).mean()  # fracción de píxeles con cambio notable
    print(f'{nombre:<12} píxeles con diferencia mayor a 10: {cambiaron:.1%}')
figura.colorbar(mapa, ax=ejes, shrink=0.7, label='diferencia (0 a 255)')
plt.show()

> **Para mirar.** Las diferencias se concentran en bordes y texturas: justo donde el modelo "agrega detalle". Un mapa así es lo que le mostrarías a una persona usuaria para que sepa **dónde confiar menos**.

## 2. Clasificación zero-shot con SigLIP

**Zero-shot** significa clasificar con categorías que escribimos nosotros, sin entrenar nada. `SigLIP` compara la imagen con cada frase y da un puntaje.

Cuidado: no fue entrenado para medir calidad. Sirve para **comparar** versiones entre sí, no como juez.

In [ ]:
from transformers import pipeline

clasificador = pipeline(
    'zero-shot-image-classification',
    model='google/siglip2-base-patch16-224',
    device=0,  # GPU
)

frases = [
    'una foto nítida y con detalle',
    'una foto borrosa',
    'una foto con mucho ruido',
    'una foto pixelada',
]

for nombre, imagen in [('original', referencia), ('bicúbica', bicubica), ('difusión x4', difusion)]:
    resultado = clasificador(imagen, candidate_labels=frases)  # lista ordenada de mayor a menor puntaje
    print(nombre)
    for fila in resultado:
        print(f"   {fila['score']:.2f}  {fila['label']}")

> **Para mirar.** Si los puntajes de "nítida" casi no cambian entre versiones, el análisis **no distingue** lo que ves a simple vista. Probá reescribir las frases: el resultado depende mucho de cómo las escribís.

## 3. Detección de objetos con RT-DETR

Un detector dibuja cajas alrededor de objetos conocidos. Lo usamos con el retrato de 1860: el original y las tres versiones con los ojos reparados. `score > 0.5` filtra las detecciones poco confiables.

In [ ]:
detector = pipeline('object-detection', model='PekingU/rtdetr_r50vd', device=0)

versiones = [('original', Image.open(f'{RESULTADOS}/retrato_original.png').convert('RGB'))]
for semilla in range(3):
    versiones.append((f'ojos seed {semilla}', Image.open(f'{RESULTADOS}/retrato_ojos_semilla{semilla}.png').convert('RGB')))

for nombre, imagen in versiones:
    detecciones = [d for d in detector(imagen) if d['score'] > 0.5]
    resumen = ', '.join(f"{d['label']} {d['score']:.2f}" for d in detecciones)
    print(f'{nombre:<14} -> {resumen if resumen else "nada detectado"}')

> **Para mirar.** Si en todas las versiones detecta la misma persona, el detector **confirma que hay una persona**, no que los ojos sean los suyos. Detectar objetos sirve para comprobar que no apareció ni desapareció algo importante. No sirve para comprobar que los detalles sean verdaderos.

## 4. Qué podemos decir y qué no

Volvé a mirar las tres secciones y completá mentalmente la tabla:

| Palabra | Pregunta | En estos ejemplos |
|---|---|---|
| **Real** | ¿Estaba en la imagen original? | Los píxeles que sobrevivieron al daño o a la reducción. |
| **Ficticio** | ¿Lo agregó el modelo? | Los ojos, las nubes, las texturas nuevas del upscaler. |
| **Creíble** | ¿Parece verdad? | Todas las versiones. |
| **Verificable** | ¿Hay evidencia externa que lo confirme? | Solo lo que comparamos con la foto de referencia (cuaderno 03). El retrato y el puente no tienen referencia. |

**Conclusión práctica:** el mapa de diferencias muestra *dónde* tocó el modelo; los modelos de análisis confirman que *el contenido general* se conserva; pero **solo una fuente externa** (otra foto, un documento, la persona) puede confirmar un detalle.

> **Para probar.** Mostrale a alguien las tres versiones del retrato sin decirle cuál es la original y preguntale cuál es "la verdadera".

## Glosario

| Término | Qué significa acá |
|---|---|
| **Zero-shot** | Clasificar con categorías definidas por vos, sin entrenar. |
| **SigLIP** | Modelo que compara imágenes con frases. |
| **RT-DETR** | Modelo que detecta objetos y dibuja cajas. |
| **Mapa de diferencias** | Imagen que muestra dónde cambió una versión respecto de otra. |
| **Verificable** | Que se puede comprobar con evidencia externa, no solo con el modelo. |